# Phase 2D - P2 One-shot Held-out (Colab)

Evaluate only the locked `P2-1S-D5` configuration. Generation is sequential. RAGAS judging is partitioned into four isolated workers, then merged only after every worker exits successfully.

- `RUN_MODE='smoke'`: five questions from the held-out **reserve**; use this only to validate plumbing.
- `RUN_MODE='heldout'`: the frozen 284-question final set from 50 unseen articles.
- This notebook never generates, judges, or scores the zero-shot P2 control.

In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='fbd2c0c791df891afa292eb56e0442e6182fe2b5'
PREPARATION_REPO_COMMIT='b9d38030e171d4a194d8e92964fae0c8b8aa597a'
HF_ARTIFACT_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-locked-v2'
HF_ARTIFACT_REVISION='locked-bge-m3-512-64-deduplicated-v2'
HF_ARTIFACT_FILENAME='artifacts/locked-bge-m3-512-64-deduplicated-v2/locked-bge-m3-512-64-deduplicated-v2.zip'
HF_ARTIFACT_SHA256='fc5d67b7acf6e8be0205ce00b8069b3b6c8dcce853f8671f2feb3887b2707a24'
HF_PREPARATION_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-experiments'
HF_PREPARATION_REVISION='771fee6e19e3e836c9ba0b15874fe0bd5199b2a5'
HF_PREPARATION_FILENAME='phase2b/preparation-refined-prompts-v1/phase2b_preparation_bundle.zip'
HF_PREPARATION_SHA256='0b2ab8388d6ab679970519bc35ba574936a063b32406ce5cee4b6ec5c166073c'
PREPARATION_BUNDLE_PATH=''  # Optional local ZIP override.
RESTORE_CHECKPOINT_PATH=''  # Optional checkpoint ZIP from this notebook.
AUTO_RESTORE_FROM_DRIVE=True
RUN_MODE='smoke'  # smoke | heldout
EXECUTE_API_CALLS=False  # Set True only after validating secrets and the locked configuration.
RUN_ID='phase2d_p2_one_shot_heldout'
PROMPT_ID='p2_1s'
CONTEXT_DEPTH=5
SMOKE_QUESTIONS=5
GENERATOR_SECRET_NAME='GEMINI_API_KEY_1'
GENERATOR_MODEL='gemini-3.1-flash-lite'
GENERATOR_REASONING_EFFORT='minimal'
GENERATOR_MAX_TOKENS=512
GENERATOR_MIN_INTERVAL_SECONDS=4.2
JUDGE_PROVIDER='fireworks'
JUDGE_MODEL='accounts/fireworks/models/glm-5p3-flash'
JUDGE_REASONING_EFFORT='low'
JUDGE_MAX_TOKENS=2048
JUDGE_WORKERS=4
CHECKPOINT_INTERVAL_SECONDS=300  # Snapshot partial judge progress to Drive every five minutes.
TOP_K=20
RERANK_TOP_N=5
SEED=42

## 1. Environment and recovery

Use a Colab T4 GPU for BGE-M3 query encoding and BGE-large reranking. Add `HF_TOKEN`, `GEMINI_API_KEY_1`, and `FIREWORKS_API_KEY` as Colab secrets. Checkpoints and final bundles are copied to `MyDrive/newsqa_phase2b`.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from google.colab import drive, userdata
drive.mount('/content/drive')
RUNTIME_ROOT=Path('/content'); DRIVE_OUTPUT_ROOT=Path('/content/drive/MyDrive/newsqa_phase2b'); DRIVE_OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
PROJECT_ROOT=RUNTIME_ROOT/'Text-Mining---NewsQA-RAG'; WORK_ROOT=RUNTIME_ROOT/RUN_ID
DATA_ROOT=WORK_ROOT/'data'; INDEX_ROOT=WORK_ROOT/'index'; RESULTS=WORK_ROOT/'results'; LOGS=WORK_ROOT/'logs'
RUNS_ROOT=WORK_ROOT/'runs'; IDS_ROOT=WORK_ROOT/'question_ids'; PROMPT_ROOT=WORK_ROOT/'prompts'; TRACE_ROOT=WORK_ROOT/'heldout_trace'
def optional_secret(name):
    try: return userdata.get(name) or ''
    except Exception: return ''
HF_TOKEN=optional_secret('HF_TOKEN'); GENERATOR_API_KEY=optional_secret(GENERATOR_SECRET_NAME); JUDGE_API_KEY=optional_secret('FIREWORKS_API_KEY')
assert RUN_MODE in {'smoke','heldout'} and PROMPT_ID=='p2_1s' and CONTEXT_DEPTH==5
assert JUDGE_WORKERS==4, 'This protocol is registered with exactly four judge shards'
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=1.20,<2'],check=True)
prep_input=Path(PREPARATION_BUNDLE_PATH) if PREPARATION_BUNDLE_PATH else None
if prep_input is None:
    assert HF_TOKEN, 'Configure the read-only HF_TOKEN Colab secret'
    from huggingface_hub import hf_hub_download
    prep_input=Path(hf_hub_download(repo_id=HF_PREPARATION_REPO_ID,repo_type='dataset',revision=HF_PREPARATION_REVISION,filename=HF_PREPARATION_FILENAME,token=HF_TOKEN))
assert hashlib.sha256(prep_input.read_bytes()).hexdigest()==HF_PREPARATION_SHA256, 'Preparation bundle checksum mismatch'
checkpoint_input=Path(RESTORE_CHECKPOINT_PATH) if RESTORE_CHECKPOINT_PATH else None
drive_checkpoint=DRIVE_OUTPUT_ROOT/f'{RUN_ID}_{RUN_MODE}_checkpoint.zip'
if checkpoint_input is None and AUTO_RESTORE_FROM_DRIVE and drive_checkpoint.exists(): checkpoint_input=drive_checkpoint
if checkpoint_input and checkpoint_input.exists():
    WORK_ROOT.mkdir(parents=True,exist_ok=True); shutil.unpack_archive(checkpoint_input,WORK_ROOT); print('Restored checkpoint:',checkpoint_input)
WORK_ROOT.mkdir(parents=True,exist_ok=True); shutil.unpack_archive(prep_input,WORK_ROOT)
for path in [DATA_ROOT,INDEX_ROOT,RESULTS,LOGS,RUNS_ROOT,IDS_ROOT,PROMPT_ROOT,TRACE_ROOT]: path.mkdir(parents=True,exist_ok=True)
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
for package_root in [PROJECT_ROOT/'common',PROJECT_ROOT/'app/backend']:
    if str(package_root) not in sys.path: sys.path.insert(0,str(package_root))
os.environ['PYTHONPATH']=os.pathsep.join([str(PROJECT_ROOT/'common'),str(PROJECT_ROOT/'app/backend'),os.environ.get('PYTHONPATH','')]).rstrip(os.pathsep)
os.environ.update({'HF_HOME':str(RUNTIME_ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false','CUDA_VISIBLE_DEVICES':'0'})
if EXECUTE_API_CALLS:
    assert GENERATOR_API_KEY, f'Configure {GENERATOR_SECRET_NAME}'
    assert JUDGE_API_KEY, 'Configure FIREWORKS_API_KEY'
print('Mode:',RUN_MODE,'| API execution:',EXECUTE_API_CALLS,'| auto checkpoint:',drive_checkpoint)

In [ ]:
import pandas as pd, yaml
from IPython.display import display
def sha256_file(path,block_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def write_json(path,value): Path(path).write_text(json.dumps(value,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def load_jsonl(path): return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]
def write_jsonl(path,records): Path(path).write_text(''.join(json.dumps(row,sort_keys=True)+'\n' for row in records),encoding='utf-8')
def latest_by_question(path):
    latest={}
    if Path(path).exists():
        for row in load_jsonl(path): latest[row['question_id']]=row
    return latest
def successful_ids(path): return {qid for qid,row in latest_by_question(path).items() if row.get('status')=='success'}
def nested(value,path):
    for part in path.split('.'):
        if not isinstance(value,dict) or part not in value: return None
        value=value[part]
    return value
def write_checkpoint():
    local=RUNTIME_ROOT/f'{RUN_ID}_{RUN_MODE}_checkpoint.zip'; temporary=local.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ['index','runs','question_ids','prompts','results','logs','heldout_trace']:
            root=WORK_ROOT/name
            if root.exists():
                for path in root.rglob('*'):
                    if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
    temporary.replace(local); shutil.copy2(local,drive_checkpoint)
    print('Checkpoint saved:',drive_checkpoint,round(drive_checkpoint.stat().st_size/2**20,1),'MiB',flush=True)
    return local
def run_command(command,label,env_overrides=None):
    command=[str(value) for value in command]; log_path=LOGS/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        env=os.environ.copy(); env.update(env_overrides or {})
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code: write_checkpoint(); raise subprocess.CalledProcessError(code,command)
    return log_path

## 2. Validate artifacts, prompt, and partitions

The final held-out IDs are loaded from the frozen Phase 2B preparation manifest. Smoke mode uses five seeded questions from `heldout_reserve`, so it cannot influence the final held-out estimate.

In [ ]:
from huggingface_hub import hf_hub_download
preparation_manifest=json.loads((RESULTS/'preparation_bundle_manifest.json').read_text())
assert preparation_manifest['repo_commit']==PREPARATION_REPO_COMMIT
for record in preparation_manifest['files']:
    path=WORK_ROOT/record['path']; assert path.exists() and path.stat().st_size==record['bytes'] and sha256_file(path)==record['sha256'],record['path']
subset_manifest=json.loads((RESULTS/'subset_manifest.json').read_text())
assert subset_manifest['heldout_outputs_accessed_for_selection'] is False
assert subset_manifest['counts']=={'development':281,'heldout':284,'heldout_reserve':587,'judge_calibration':20,'screening':80,'smoke':5}
for name,digest in subset_manifest['sha256'].items(): assert sha256_file(IDS_ROOT/f'{name}.json')==digest,name
artifact_root=DATA_ROOT/'locked-bge-m3-512-64-deduplicated-v2'
if not (artifact_root/'bundle_manifest.json').exists():
    artifact_zip=Path(hf_hub_download(repo_id=HF_ARTIFACT_REPO_ID,repo_type='dataset',revision=HF_ARTIFACT_REVISION,filename=HF_ARTIFACT_FILENAME,token=HF_TOKEN or None))
    assert sha256_file(artifact_zip)==HF_ARTIFACT_SHA256; artifact_root.mkdir(parents=True,exist_ok=True); shutil.unpack_archive(artifact_zip,artifact_root)
bundle_manifest=json.loads((artifact_root/'bundle_manifest.json').read_text())
assert bundle_manifest['statistics']['chunks']==22766 and bundle_manifest['statistics']['resolved_questions']==1152
for relative,record in bundle_manifest['artifacts'].items():
    path=artifact_root/relative; assert path.exists() and path.stat().st_size==record['bytes'] and sha256_file(path)==record['sha256'],relative
testset=artifact_root/'testset_resolved.jsonl'; chunks=artifact_root/'chunks.jsonl'; sparse_index=artifact_root/'bge_m3_sparse.pkl'
test_rows={row['question_id']:row for row in load_jsonl(testset)}
development_ids=json.loads((IDS_ROOT/'development.json').read_text()); heldout_ids=json.loads((IDS_ROOT/'heldout.json').read_text()); reserve_ids=json.loads((IDS_ROOT/'heldout_reserve.json').read_text())
assert len(heldout_ids)==284 and len(reserve_ids)==587 and set(heldout_ids).isdisjoint(development_ids)
heldout_articles={test_rows[qid]['article_key'] for qid in heldout_ids}; assert len(heldout_articles)==50
smoke_ids=sorted(reserve_ids,key=lambda qid:hashlib.sha256(f'{SEED}:heldout-smoke:{qid}'.encode()).hexdigest())[:SMOKE_QUESTIONS]
active_ids=smoke_ids if RUN_MODE=='smoke' else heldout_ids
active_partition='heldout_reserve_smoke' if RUN_MODE=='smoke' else 'heldout'
active_ids_path=IDS_ROOT/f'{active_partition}.json'; write_json(active_ids_path,active_ids)
assert set(active_ids).isdisjoint(development_ids)
print('Partition:',active_partition,'| questions:',len(active_ids),'| articles:',len({test_rows[qid]['article_key'] for qid in active_ids}))

In [ ]:
from newsqa_rag.evaluation.benchmark_io import stable_hash
config=yaml.safe_load((PROJECT_ROOT/'configs/config.yaml').read_text())
config['chunking'].update({'strategy':'recursive','chunk_size':512,'chunk_overlap':64})
config['llm'].update({'model':GENERATOR_MODEL,'temperature':0.0,'max_tokens':GENERATOR_MAX_TOKENS,'reasoning_effort':GENERATOR_REASONING_EFFORT})
config['retrieval'].update({'retriever':'sparse','top_k':TOP_K})
config['retrieval']['sparse'].update({'method':'bge-m3','model':'BAAI/bge-m3','device':'cuda'})
config['retrieval']['reranker'].update({'enabled':True,'type':'cross-encoder','model':'BAAI/bge-reranker-large','top_n':RERANK_TOP_N,'batch_size':8,'device':'cuda'})
config_path=INDEX_ROOT/'heldout_config.yaml'; config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
profile=json.loads((INDEX_ROOT/'phase2b_variant.json').read_text())
profile['pipeline'].update({'config_path':str(config_path),'config_sha256':stable_hash(config)})
profile['database'].update({'indexed':False,'chunk_count':22766})
profile['artifacts']['chunks']={'path':str(chunks),'bytes':chunks.stat().st_size,'sha256':sha256_file(chunks)}
profile['artifacts']['testset_resolved']={'path':str(testset),'bytes':testset.stat().st_size,'sha256':sha256_file(testset)}
profile['artifacts']['bm25']={'path':str(sparse_index),'bytes':sparse_index.stat().st_size,'sha256':sha256_file(sparse_index)}
profile_path=INDEX_ROOT/'heldout_variant.json'; write_json(profile_path,profile)
prompt_registry=yaml.safe_load((PROJECT_ROOT/'configs/experiments/phase2_generation_prompts.yaml').read_text())['prompts']
assert 'p2' in prompt_registry
one_shot_demonstration=("\n\nDemonstration (format example only):\nContext [1]: The Northbridge museum opened a new gallery in 2018.\nContext [2]: The city council approved the Riverside transit plan in March 2021.\nQuestion: When was the Riverside transit plan approved?\nAnswer: March 2021 [2]\n\nNow answer the user's question using the supplied numbered context and the same concise citation format.")
one_shot_prompt=prompt_registry['p2']['system_prompt']+one_shot_demonstration
prompt_path=PROMPT_ROOT/'p2_1s.txt'; prompt_path.write_text(one_shot_prompt,encoding='utf-8')
prompt_contract={'schema_version':1,'prompt_id':'p2_1s','parent_prompt_id':'p2','demonstration_source':'synthetic_fixed_v1','system_prompt_sha256':sha256_file(prompt_path),'context_depth':5}
write_json(RESULTS/'one_shot_heldout_contract.json',prompt_contract)
print('P2-1S SHA-256:',prompt_contract['system_prompt_sha256'])

## 3. Sequential retrieval and generation

There is exactly one collector process. The generator is rate-spaced and processes questions sequentially. A checkpoint is saved after retrieval and again after generation.

In [ ]:
assert EXECUTE_API_CALLS, 'Set EXECUTE_API_CALLS=True after checking RUN_MODE and secrets'
import torch
assert torch.cuda.is_available(), 'Enable a Colab GPU for retrieval and reranking'
trace_dir=TRACE_ROOT/active_partition; trace_dir.mkdir(parents=True,exist_ok=True)
retrievals=trace_dir/'retrievals.jsonl'
retrieval_command=[sys.executable,'-u','scripts/collect_benchmark_predictions.py','--retriever','sparse','--reranker','cross-encoder','--reranker-model','BAAI/bge-reranker-large','--testset',testset,'--variant-manifest',profile_path,'--config',config_path,'--run-dir',trace_dir,'--question-ids-file',active_ids_path,'--top-k',TOP_K,'--rerank-top-n',RERANK_TOP_N,'--seed',SEED,'--retrieval-only','--max-attempts',3,'--retry-failed','--progress']
run_command(retrieval_command,f'{RUN_MODE}_retrieval')
assert successful_ids(retrievals)==set(active_ids), 'Retrieval is incomplete'
write_checkpoint()
run_dir=RUNS_ROOT/f'{active_partition}__p2_1s__d5'; run_dir.mkdir(parents=True,exist_ok=True)
generation_command=[sys.executable,'-u','scripts/collect_benchmark_predictions.py','--retriever','sparse','--reranker','cross-encoder','--reranker-model','BAAI/bge-reranker-large','--testset',testset,'--variant-manifest',profile_path,'--config',config_path,'--run-dir',run_dir,'--question-ids-file',active_ids_path,'--top-k',TOP_K,'--rerank-top-n',RERANK_TOP_N,'--generator-model',GENERATOR_MODEL,'--prompt-id','p2_1s','--system-prompt-file',prompt_path,'--context-depth',5,'--source-retrievals',retrievals,'--generation-min-interval-seconds',GENERATOR_MIN_INTERVAL_SECONDS,'--seed',SEED,'--max-attempts',3,'--retry-failed','--progress']
run_command(generation_command,f'{RUN_MODE}_generation_sequential',{'GEMINI_API_KEY':GENERATOR_API_KEY})
assert successful_ids(run_dir/'predictions.jsonl')==set(active_ids), 'Generation is incomplete'
run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',run_dir],f'{RUN_MODE}_score_prejudge')
write_checkpoint()
print('Sequential generation complete:',len(active_ids),'questions')

## 4. Launch four isolated judge workers

Each question belongs to one deterministic shard. Every worker writes separate results, attempts, and logs. Do not rerun the generation cell while these processes are active.

In [ ]:
assert successful_ids(run_dir/'predictions.jsonl')==set(active_ids)
judge_shards=[active_ids[index::JUDGE_WORKERS] for index in range(JUDGE_WORKERS)]
assert sum(len(shard) for shard in judge_shards)==len(active_ids)
assert set().union(*map(set,judge_shards))==set(active_ids)
assert sum(len(set(a)&set(b)) for i,a in enumerate(judge_shards) for b in judge_shards[i+1:])==0
judge_processes=[]
for worker_index,shard in enumerate(judge_shards,1):
    ids_path=IDS_ROOT/f'judge_{active_partition}_worker_{worker_index}.json'; write_json(ids_path,shard)
    results_file=f'judge_results_worker_{worker_index}.jsonl'; attempts_file=f'judge_attempts_worker_{worker_index}.jsonl'
    log_path=LOGS/f'{RUN_MODE}_judge_worker_{worker_index}_{time.strftime("%Y%m%d_%H%M%S")}.log'; log_handle=log_path.open('a',encoding='utf-8')
    command=[sys.executable,'-u','scripts/judge_benchmark_predictions.py','--run-dir',run_dir,'--judge-provider',JUDGE_PROVIDER,'--judge-model',JUDGE_MODEL,'--reasoning-effort',JUDGE_REASONING_EFFORT,'--judge-max-tokens',JUDGE_MAX_TOKENS,'--question-ids-file',ids_path,'--results-file',results_file,'--attempts-file',attempts_file,'--batch-size',1,'--max-workers',1,'--seed',SEED+worker_index,'--max-attempts',3,'--retry-failed','--require-complete-metrics','--progress']
    env=os.environ.copy(); env['FIREWORKS_API_KEY']=JUDGE_API_KEY
    process=subprocess.Popen([str(x) for x in command],cwd=PROJECT_ROOT,env=env,stdout=log_handle,stderr=subprocess.STDOUT,text=True)
    judge_processes.append({'worker':worker_index,'process':process,'log_handle':log_handle,'log_path':log_path,'ids':shard,'results_file':results_file,'attempts_file':attempts_file})
    print(f'Worker {worker_index}: PID={process.pid}, questions={len(shard)}, log={log_path}')
print('All four judge workers started. Run the next cell to wait, validate, and merge.')

## 5. Wait for workers, merge, and score

This cell blocks until all four workers exit. It keeps only the latest successful record per assigned question, verifies complete/disjoint coverage, writes the canonical `judge_results.jsonl`, then scores the run.

In [ ]:
assert 'judge_processes' in globals() and len(judge_processes)==JUDGE_WORKERS, 'Run the launch cell first'
worker_status=[]
try:
    last_checkpoint=time.monotonic()
    while any(worker['process'].poll() is None for worker in judge_processes):
        time.sleep(5)
        if time.monotonic()-last_checkpoint>=CHECKPOINT_INTERVAL_SECONDS:
            write_checkpoint(); last_checkpoint=time.monotonic()
            running=[worker['worker'] for worker in judge_processes if worker['process'].poll() is None]
            print('Judge workers still running:',running,flush=True)
    for worker in judge_processes:
        code=worker['process'].wait(); worker['log_handle'].close()
        worker_status.append({'worker':worker['worker'],'exit_code':code,'questions':len(worker['ids']),'log':str(worker['log_path'])})
        print(f"Worker {worker['worker']} exited with {code}")
    failed=[row for row in worker_status if row['exit_code']!=0]
    if failed: raise RuntimeError(f'Judge workers failed: {failed}')
    merged=[]; seen=set()
    for worker in judge_processes:
        records=latest_by_question(run_dir/worker['results_file']); expected=set(worker['ids'])
        successful={qid:row for qid,row in records.items() if row.get('status')=='success'}
        assert set(successful)==expected,f"Worker {worker['worker']} incomplete: {sorted(expected-set(successful))[:5]}"
        assert seen.isdisjoint(successful),f"Duplicate questions in worker {worker['worker']}"
        seen.update(successful); merged.extend(successful[qid] for qid in worker['ids'])
    assert seen==set(active_ids) and len(merged)==len(active_ids)
    write_jsonl(run_dir/'judge_results.jsonl',merged)
    write_json(RESULTS/f'{active_partition}_judge_shards.json',{'schema_version':1,'workers':JUDGE_WORKERS,'partition':active_partition,'question_count':len(active_ids),'status':'complete','shards':[{'worker':worker['worker'],'questions':len(worker['ids']),'ids_sha256':sha256_file(IDS_ROOT/f"judge_{active_partition}_worker_{worker['worker']}.json"),'results_file':worker['results_file'],'results_sha256':sha256_file(run_dir/worker['results_file'])} for worker in judge_processes]})
    run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',run_dir],f'{RUN_MODE}_score_final')
    assert successful_ids(run_dir/'judge_results.jsonl')==set(active_ids)
except Exception:
    write_checkpoint(); raise
write_checkpoint()
display(pd.DataFrame(worker_status))
print('Merged judge results:',run_dir/'judge_results.jsonl')

## 6. Report and export

Held-out results estimate generalization only. Do not use them to revise P2-1S-D5. Smoke output is a systems check and must not be reported as model performance.

In [ ]:
report=json.loads((run_dir/'report.json').read_text())
expected=len(active_ids)
assert nested(report,'coverage.expected')==expected and nested(report,'coverage.successful')==expected
assert nested(report,'ragas.n_samples')==expected
summary={'schema_version':1,'partition':active_partition,'run_mode':RUN_MODE,'articles':len({test_rows[qid]['article_key'] for qid in active_ids}),'questions':expected,'configuration':{'prompt_id':'p2_1s','parent_prompt_id':'p2','context_depth':5,'retriever':'bge-m3-sparse','top_k':TOP_K,'reranker':'BAAI/bge-reranker-large','rerank_top_n':RERANK_TOP_N,'generator_model':GENERATOR_MODEL,'generator_reasoning_effort':GENERATOR_REASONING_EFFORT,'judge_provider':JUDGE_PROVIDER,'judge_model':JUDGE_MODEL,'judge_reasoning_effort':JUDGE_REASONING_EFFORT,'judge_workers':JUDGE_WORKERS},'metrics':{'exact_match':nested(report,'qa.exact_match'),'token_f1':nested(report,'qa.f1'),'answer_correctness':nested(report,'ragas.answer_correctness'),'faithfulness':nested(report,'ragas.faithfulness'),'answer_relevancy':nested(report,'ragas.answer_relevancy'),'context_precision':nested(report,'ragas.context_precision'),'context_recall':nested(report,'ragas.context_recall'),'citation_f1':nested(report,'citations.citation_f1'),'citation_validity':nested(report,'citations.citation_validity')},'coverage':report['coverage'],'provenance':{'repo_commit':REPO_COMMIT,'artifact_sha256':HF_ARTIFACT_SHA256,'preparation_sha256':HF_PREPARATION_SHA256,'active_ids_sha256':sha256_file(active_ids_path),'prompt_sha256':sha256_file(prompt_path),'run_fingerprint':json.loads((run_dir/'run_manifest.json').read_text())['run_fingerprint']},'completed_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
write_json(RESULTS/f'{active_partition}_summary.json',summary); pd.json_normalize(summary,sep='.').to_csv(RESULTS/f'{active_partition}_summary.csv',index=False)
display(pd.DataFrame([summary['metrics']]))
checkpoint=write_checkpoint()
result_bundle=RUNTIME_ROOT/f'{RUN_ID}_{RUN_MODE}_results.zip'; temporary=result_bundle.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['runs','question_ids','prompts','results','logs','heldout_trace']:
        root=WORK_ROOT/name
        if root.exists():
            for path in root.rglob('*'):
                if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
temporary.replace(result_bundle)
for artifact in [checkpoint,result_bundle]:
    target=DRIVE_OUTPUT_ROOT/Path(artifact).name
    if Path(artifact).resolve()!=target.resolve(): shutil.copy2(artifact,target)
    print('Saved:',target,round(target.stat().st_size/2**20,1),'MiB')